## Скачиваем модель для перевод текста с русского на английский

Мы могли бы использовать апи для перевода но у этого подхода есть несколько минусов
1. Наш сервис будет зависеть от интернет соединения
2. При проблемах с АПИ(а они бывают часто потому что апи гугл переводичка не официальное и часто отваливается) сервис становится не доступным

Поэтому мы приняли решение скачать модель для перевода Helsinki-NLP/opus-mt-en-ru и запустить её локально, вот плюсы этого решения
1. Наш сервис не будет зависеть от интернет соединения, и будет доступен офлайн
2. Мы не зависим от внешних сервисов

In [7]:
import os
import time

import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer
from huggingface_hub.utils import enable_progress_bars
from tqdm import tqdm

enable_progress_bars()

DATA_DIR = "../DATA"
MODELS_DIR = "../MODELS"

INPUT_CSV = os.path.join(DATA_DIR, "captions.csv")
OUTPUT_CSV = os.path.join(DATA_DIR, "captions_ru.csv")

MODEL_ID = "Helsinki-NLP/opus-mt-en-ru"
MODEL_DIR = os.path.join(MODELS_DIR, "opus-mt-en-ru")

BATCH = 64

торч 2.13.0+cpu


In [ ]:
device = "cpu"
df = pd.read_csv(INPUT_CSV)
df.head()

## Проверка данных

Видим что не на все фото 5 описаний, поэтому кол-во записей больше 25000

In [10]:
df.groupby("image_id").size().value_counts()

5    4987
6      10
4       2
7       1
Name: count, dtype: int64

## Качаем модель

Качаем opus-mt-en-ru с Hugging Face и сохраняем в MODELS.


In [11]:
tokenizer = MarianTokenizer.from_pretrained(MODEL_ID)
model = MarianMTModel.from_pretrained(MODEL_ID)
tokenizer.save_pretrained(MODEL_DIR)
model.save_pretrained(MODEL_DIR)

model = model.to(device)
model.eval()

качаю Helsinki-NLP/opus-mt-en-ru


/home/decrich/PycharmProjects/PythonProject/.venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


KeyboardInterrupt: 

In [ ]:
def translate(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_length=128)
    return tokenizer.batch_decode(out, skip_special_tokens=True)

## Проверяем на примерах

Перед тем как гнать 25 тысяч строк смотрим глазами что модель вообще работает и переводит осмысленно. Проверка дешевая, а если модель выдает мусор то лучше узнать это сейчас чем через час.

In [ ]:
texts = df["caption_en"].head(5).tolist()

for en, ru in zip(texts, translate(texts)):
    print("EN:", en)
    print("RU:", ru)
    print()

## Переводим только уникальные подписи

При анализе мы увидели что из 25 тысяч подписей уникальных 24808, около 200 фраз повторяются. Переводить одно и то же дважды смысла нет

In [ ]:
uniq = df["caption_en"].unique().tolist()

print("строк в таблице", len(df))
print("уникальных подписей", len(uniq))
print("экономим переводов", len(df) - len(uniq))
print("батчей будет", (len(uniq) + BATCH - 1) // BATCH)

## Основной расчет


Каждые 50 батчей сохраняем сохраняем переведенный текст в  временный файл. Это нужно чтобы есди код упадет(а он впоне может, мы это поняли)на 380 батче из 388 то не хочется терять всю работу.

In [ ]:
trans = []
start = time.time()

for i in tqdm(range(0, len(uniq), BATCH)):
    trans.extend(translate(uniq[i:i + BATCH]))

    if i > 0 and i % (BATCH * 50) == 0:
        pd.DataFrame({
            "caption_en": uniq[:len(trans)],
            "caption_ru": trans,
        }).to_csv(os.path.join(DATA_DIR, "_checkpoint.csv"), index=False)

spent = time.time() - start

In [ ]:
d = dict(zip(uniq, trans))

df["caption_ru"] = df["caption_en"].map(d)

In [ ]:
for _, row in df.sample(8, random_state=42).iterrows():
    print(row["file_name"])
    print("  EN:", row["caption_en"])
    print("  RU:", row["caption_ru"])
    print()

In [ ]:
print("пустых переводов", (df["caption_ru"].str.strip() == "").sum())
print("средняя длина EN %.1f символов" % df["caption_en"].str.len().mean())
print("средняя длина RU %.1f символов" % df["caption_ru"].str.len().mean())
print("перевод совпал с оригиналом", (df["caption_ru"] == df["caption_en"]).sum())

In [ ]:
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")